In [5]:
import pandas as pd

# 1. Đọc file CSV từ thư mục data/raw/
file_path = '../data/raw/scanner_data.csv'
df = pd.read_csv(file_path)

# 2. In ra 5 dòng đầu tiên để xem "mặt mũi" dữ liệu
print("=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===")
display(df.head())

# 3. Xem tổng quan kích thước (Bao nhiêu dòng, bao nhiêu cột, có cột nào bị rỗng không)
print("\n=== THÔNG TIN TỔNG QUAN ===")
df.info()

=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===


,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
0,1,02/01/2016,2547,1,X52,0EM7L,1.0,3.13
1,2,02/01/2016,822,2,2ML,68BRQ,1.0,5.46
2,3,02/01/2016,3686,3,0H2,CZUZX,1.0,6.35
3,4,02/01/2016,3719,4,0H2,549KK,1.0,5.59
4,5,02/01/2016,9200,5,0H2,K8EHH,1.0,6.88



=== THÔNG TIN TỔNG QUAN ===
<class 'pandas.DataFrame'>
RangeIndex: 131706 entries, 0 to 131705
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Unnamed: 0      131706 non-null  int64  
 1   Date            131706 non-null  str    
 2   Customer_ID     131706 non-null  int64  
 3   Transaction_ID  131706 non-null  int64  
 4   SKU_Category    131706 non-null  str    
 5   SKU             131706 non-null  str    
 6   Quantity        131706 non-null  float64
 7   Sales_Amount    131706 non-null  float64
dtypes: float64(2), int64(3), str(3)
memory usage: 8.0 MB


In [6]:
# 1. In ra tên của 8 cột để xem siêu thị quản lý những thông tin gì
print("=== DANH SÁCH CÁC CỘT ===")
print(df.columns.tolist())

# 2. Hiển thị 5 dòng đầu tiên dưới dạng bảng Excel cho dễ nhìn
print("\n=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===")
display(df.head())

# 3. Quét xem có dòng nào bị thiếu dữ liệu (NULL/NaN) không để lát dọn dẹp
print("\n=== SỐ LƯỢNG DỮ LIỆU BỊ TRỐNG ===")
print(df.isnull().sum())

=== DANH SÁCH CÁC CỘT ===
['Unnamed: 0', 'Date', 'Customer_ID', 'Transaction_ID', 'SKU_Category', 'SKU', 'Quantity', 'Sales_Amount']

=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===


,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
0,1,02/01/2016,2547,1,X52,0EM7L,1.0,3.13
1,2,02/01/2016,822,2,2ML,68BRQ,1.0,5.46
2,3,02/01/2016,3686,3,0H2,CZUZX,1.0,6.35
3,4,02/01/2016,3719,4,0H2,549KK,1.0,5.59
4,5,02/01/2016,9200,5,0H2,K8EHH,1.0,6.88



=== SỐ LƯỢNG DỮ LIỆU BỊ TRỐNG ===
Unnamed: 0        0
Date              0
Customer_ID       0
Transaction_ID    0
SKU_Category      0
SKU               0
Quantity          0
Sales_Amount      0
dtype: int64


In [16]:
from sklearn.preprocessing import LabelEncoder

# 1. Copy data để nhào nặn, không làm hỏng file gốc
df_clean = df.copy()

# 2. Vứt bỏ các cột rác không có giá trị dự báo
columns_to_drop = ['Unnamed: 0', 'Customer_ID', 'Transaction_ID', 'Sales_Amount']
df_clean = df_clean.drop(columns=columns_to_drop)

# 3. Băm nhỏ cột Ngày tháng (ĐÃ FIX LỖI VALUE ERROR)
# Ép kiểu an toàn: Ưu tiên ngày trước tháng, lỗi thì bỏ qua
df_clean['Date'] = pd.to_datetime(df_clean['Date'], dayfirst=True, errors='coerce')

# Dọn dẹp sạch sẽ các dòng bị lỗi ngày tháng (nếu có) để AI không bị "ngộ độc"
df_clean = df_clean.dropna(subset=['Date'])

df_clean['Month'] = df_clean['Date'].dt.month       # Lấy Tháng
df_clean['DayOfWeek'] = df_clean['Date'].dt.dayofweek # Lấy Thứ (0=Thứ 2, 6=Chủ nhật)
df_clean = df_clean.drop(columns=['Date'])          # Băm xong thì xóa cột gốc

# 4. Mã hóa các cột dạng Chữ thành Số (Label Encoding)
le_category = LabelEncoder()
df_clean['SKU_Category'] = le_category.fit_transform(df_clean['SKU_Category'])

le_sku = LabelEncoder()
df_clean['SKU'] = le_sku.fit_transform(df_clean['SKU'])

# 5. In ra thành quả
print("=== DỮ LIỆU ĐÃ LÀM SẠCH - SẴN SÀNG CHO AI HỌC ===")
display(df_clean.head())

=== DỮ LIỆU ĐÃ LÀM SẠCH - SẴN SÀNG CHO AI HỌC ===


,SKU_Category,SKU,Quantity,Month,DayOfWeek
0,174,63,1.0,1,5
1,17,922,1.0,1,5
2,2,1932,1.0,1,5
3,2,776,1.0,1,5
4,2,2971,1.0,1,5


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib # Thư viện dùng để đóng gói "não bộ"
import os

# 1. Tách Dữ liệu thành "Câu hỏi" (X) và "Đáp án" (y)
# Câu hỏi: Hôm nay bán loại hàng gì, mã số mấy, tháng mấy, thứ mấy?
X = df_clean.drop(columns=['Quantity'])
# Đáp án: Số lượng bán được là bao nhiêu?
y = df_clean['Quantity']

# 2. Chia bài: Lấy 80% dữ liệu cho AI học, 20% để kiểm tra (thi học kỳ)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Triệu hồi và Huấn luyện AI
print("🤖 Đang đưa AI vào buồng huấn luyện... Quá trình này mất khoảng 5-10 giây...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train) # Lệnh này là lệnh "HỌC"

# 4. Cho AI làm bài thi và chấm điểm
y_pred = model.predict(X_test)
sai_so = mean_absolute_error(y_test, y_pred)
diem_r2 = r2_score(y_test, y_pred)

print(f"\n✅ HUẤN LUYỆN THÀNH CÔNG!")
print(f"📉 Sai số dự báo trung bình: Lệch khoảng {sai_so:.2f} sản phẩm")
print(f"🎯 Điểm thông minh (R2 Score): {diem_r2:.2f} (Càng gần 1.0 là AI càng giỏi)")

# 5. Xuất "não bộ" ra file để đem đi xài cho phần mềm chính
os.makedirs('../models', exist_ok=True) # Tự động tạo thư mục models nếu chưa có
joblib.dump(model, '../models/price_forecast.pkl')
print("\n💾 ĐÃ LƯU FILE NÃO BỘ VÀO: models/price_forecast.pkl")

🤖 Đang đưa AI vào buồng huấn luyện... Quá trình này mất khoảng 5-10 giây...

✅ HUẤN LUYỆN THÀNH CÔNG!
📉 Sai số dự báo trung bình: Lệch khoảng 0.46 sản phẩm
🎯 Điểm thông minh (R2 Score): 0.50 (Càng gần 1.0 là AI càng giỏi)

💾 ĐÃ LƯU FILE NÃO BỘ VÀO: models/price_forecast.pkl


In [19]:
import joblib

# Lưu cuốn từ điển dịch Loại hàng (Category)
joblib.dump(le_category, '../models/encoder_category.pkl')

# Lưu cuốn từ điển dịch Mã hàng (SKU)
joblib.dump(le_sku, '../models/encoder_sku.pkl')

print("📚 ĐÃ LƯU 2 CUỐN TỪ ĐIỂN VÀO THƯ MỤC models/")

📚 ĐÃ LƯU 2 CUỐN TỪ ĐIỂN VÀO THƯ MỤC models/


In [22]:
# 1. In ra toàn bộ các loại hàng (Category) mà cuốn từ điển đã ghi nhớ
print("=== 1. TẤT CẢ CÁC TỪ VỰNG TRONG TỪ ĐIỂN (SKU_Category) ===")
danh_sach_loai_hang = le_category.classes_
print(danh_sach_loai_hang[:10]) # Chỉ in thử 10 món đầu tiên cho đỡ dài

# 2. Test dịch từ CHỮ sang SỐ (Mô phỏng lúc người dùng thao tác trên Giao diện)
chu_can_dich = [danh_sach_loai_hang[0], danh_sach_loai_hang[1]] # Lấy thử 2 món đầu tiên
so_da_dich = le_category.transform(chu_can_dich)

print("\n=== 2. PHẦN MỀM DỊCH CHỮ SANG SỐ ĐỂ CHO AI HIỂU ===")
for chu, so in zip(chu_can_dich, so_da_dich):
    print(f"Người dùng chọn '{chu}' ---> Gửi vào AI là số: {so}")

# 3. Test dịch ngược từ SỐ về CHỮ (Mô phỏng lúc lấy từ CSDL lên Giao diện)
so_can_test = [0, 1, 2] # Thử hỏi từ điển xem số 0, 1, 2 là cái gì
chu_dich_nguoc = le_category.inverse_transform(so_can_test)

print("\n=== 3. PHẦN MỀM DỊCH SỐ VỀ CHỮ ĐỂ HIỂN THỊ CHO NGƯỜI DÙNG ===")
for so, chu in zip(so_can_test, chu_dich_nguoc):
    print(f"AI nhả ra số {so} ---> Dịch lên màn hình là: '{chu}'")

=== 1. TẤT CẢ CÁC TỪ VỰNG TRONG TỪ ĐIỂN (SKU_Category) ===
['01F' '06Z' '0H2' '0KX' '0WT' '10Y' '144' '1EO' '1L6' '1R3']

=== 2. PHẦN MỀM DỊCH CHỮ SANG SỐ ĐỂ CHO AI HIỂU ===
Người dùng chọn '01F' ---> Gửi vào AI là số: 0
Người dùng chọn '06Z' ---> Gửi vào AI là số: 1

=== 3. PHẦN MỀM DỊCH SỐ VỀ CHỮ ĐỂ HIỂN THỊ CHO NGƯỜI DÙNG ===
AI nhả ra số 0 ---> Dịch lên màn hình là: '01F'
AI nhả ra số 1 ---> Dịch lên màn hình là: '06Z'
AI nhả ra số 2 ---> Dịch lên màn hình là: '0H2'
